# 01 — Charades → ADL skeleton shards (**OFFLINE**, RTX PRO 6000 Blackwell, NO INTERNET)

Runs RTMO over the staged Charades archive at 15 fps (the deployment rate: extracting at
a different rate than serving is a train/serve skew), maps the 157 classes through the
reviewed taxonomy map, windows into 2 s / stride 1 s, and writes fp16 `.npz` shards in
the layout `SkeletonWindowDataset` loads. The download happened once in notebook 01a;
nothing here touches the network, so the whole 12 h goes to inference on the Blackwell.

| attach as input | produces |
|---|---|
| `behaviorsense-code` | `behaviorsense-adl-shards` |
| the wheels dataset (e.g. `behavioursense-WW`) | (updated version of the same) |
| `charades-480p` (from notebook 01a) | |
| `behaviorsense-adl-shards` (**continuation runs only**) | |

**One session DOES finish this — measured, on the Blackwell with CUDA confirmed.** The
extraction loop reported **1333 videos/h**, and this notebook reads `Charades_v1_train.csv`
(**7,985** rows — the 9,848 figure is train + test, and only train is labelled), so a full
pass is **~6 h** against the 12 h limit. Output is ~232k windows, ~0.8 GB compressed,
nowhere near the 20 GB `/kaggle/working` cap.

So set `VID_END` past the end of the split (`99999` — Python slicing clamps) and do it in
one run. The earlier `3000` default predates the measurement and cost nothing but three
extra sessions' worth of setup.

The slicing machinery stays, because it is what makes a *timeout* survivable rather than
what makes the job fit: shards flush every 400 MB (~1,100 videos), so an interrupted
session keeps everything already written, and the next run resumes at
`VID_START=<where you stopped>` with the carry-forward cell bringing prior shards forward.
Kaggle dataset versions REPLACE content — that cell is what makes accumulation work.

⚠️ **Quick Save, not "Save & Run All".** Save & Run All re-executes from scratch and would
spend another 6 h re-extracting. Quick Save snapshots `/kaggle/working` as it stands.

In [ ]:
# Whole train split in one session: 7,985 rows / 1333 videos-per-h ~= 6 h < 12 h.
# Slicing clamps, so 99999 means "to the end" and needs no exact row count here.
# Set VID_START to where a timed-out session stopped; shards already flushed are kept.
VID_START, VID_END = 0, 99999
FPS_SAMPLE = 15
import subprocess, sys, pathlib

# OFFLINE install from the staged wheels. --no-index is what makes "offline" true rather
# than aspirational: without it pip reaches for PyPI, hangs on a dead socket, and the
# failure reads as a package problem instead of a network one.
INPUT = pathlib.Path("/kaggle/input")

# Pick the wheel cache by CONTENT. /kaggle/input also holds attached competitions, and at
# least one (arc-prize-2026) ships .whl files - "first *.whl found" selected that one.
_MARKERS = {"torch", "rtmlib", "onnxruntime-gpu", "nvidia-cudnn-cu12", "triton"}
_dirs = {}
for _w in INPUT.glob("**/*.whl"):
    _dirs.setdefault(_w.parent, set()).add(_w.name.split("-")[0].lower().replace("_", "-"))
assert _dirs, f"no wheels attached. Mounted: {sorted(p.name for p in INPUT.iterdir())}"
WHEEL_DIR, _hits = max(_dirs.items(), key=lambda kv: len(kv[1] & _MARKERS))
assert _hits & _MARKERS, (
    f"no staged wheel cache found - candidates hold none of {sorted(_MARKERS)}: "
    f"{[str(d) for d in _dirs]}")

# rtmlib DEPENDS ON onnxruntime (the CPU build). Installing it normally therefore drags
# the CPU package in, and pip installs it AFTER onnxruntime-gpu - both own the same
# `onnxruntime` module directory, so the CPU build overwrites the GPU one's provider
# registration. Observed exactly this: "Successfully installed onnxruntime-1.28.0
# onnxruntime-gpu-1.28.0 rtmlib-0.0.16", then providers = [Azure, CPU] and no CUDA.
#
# So: remove any CPU build, install rtmlib WITHOUT its deps (numpy/opencv/tqdm are all in
# the Kaggle image already), and install onnxruntime-gpu LAST so nothing can clobber it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                "--find-links", str(WHEEL_DIR), "rtmlib"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                "--find-links", str(WHEEL_DIR),
                "onnxruntime-gpu", "opencv-python-headless", "PyYAML"], check=True)
print(f"installed from {WHEEL_DIR}")

In [ ]:
# Resolve the repo mount by CONTENT, not by dataset name.
#
# The dataset title is free text and this project has already been uploaded under more
# than one spelling ("behaviorsense-*" and "behavioursense-*"). Hard-coding the name makes
# cell 1 of a 12-hour session fail on a typo, so find the repo by a file only it contains.
import pathlib

INPUT = pathlib.Path("/kaggle/input")
def attached_mounts():
    # Datasets do NOT sit directly under /kaggle/input. They mount at
    # /kaggle/input/datasets/<owner>/<name>/, and competitions at
    # /kaggle/input/competitions/<name>/. Listing INPUT.iterdir() therefore always
    # reports ['competitions', 'datasets'] whatever is attached - which is what the
    # "code: nothing matches ..." failure printed, telling us nothing about whether
    # the code dataset was attached. Descend to the level that names real mounts, and
    # report what each one CONTAINS, since a dataset can be attached and still be
    # missing the directory the notebook needs.
    out = []
    for container in ("datasets", "competitions"):
        base = INPUT / container
        if not base.is_dir():
            continue
        for owner in sorted(base.iterdir()):
            kids = sorted(owner.iterdir()) if owner.is_dir() else []
            if kids and all(k.is_dir() for k in kids[:1]) and container == "datasets":
                for ds in kids:
                    top = sorted(q.name for q in ds.iterdir())[:6] if ds.is_dir() else []
                    out.append(f"{ds.name} (top level: {top})")
            else:
                out.append(owner.name)
    # Fall back to the flat layout so this keeps working if Kaggle changes the mount
    # shape back, rather than reporting nothing at all.
    return out or sorted(p.name for p in INPUT.iterdir())

ATTACHED = attached_mounts() if INPUT.is_dir() else []
_init = sorted(INPUT.glob("**/src/behaviorsense/__init__.py"))
assert _init, (f"repo dataset not found: nothing matches **/src/behaviorsense/__init__.py "
               f"under /kaggle/input. Attached datasets: {ATTACHED}")
SRC     = _init[0].parent.parent
CODE    = SRC.parent
SCRIPTS = CODE / "scripts"
CONFIGS = CODE / "configs"
import sys
sys.path.insert(0, str(SRC)); sys.path.insert(0, str(SCRIPTS))
print(f"  code {CODE}")
print(f"  attached {ATTACHED}")

CONTRACT = [
    ("scripts/kaggle_smoke_test.py", "--profile",      "notebook 03 preflight"),
    ("scripts/train_adl.py",         "--stop-after",   "resume guard (notebook 03)"),
    ("scripts/train_fall.py", "pos_rate > 0.5 and args.focal_alpha > 0.5",
     "refuses focal alpha that up-weights the majority; a snapshot without it trains "
     "the fall head on 82% positives and reports a plausible but meaningless AUPRC"),
    ("scripts/prepare_skeletons.py", "def assign_slots",
     "slot tracking + windowing (notebooks 01/02)"),
    ("scripts/prepare_skeletons.py", "with_starts",
     "fall labelling by true frame position (notebook 02); index-derived position "
     "mislabels the descent whenever a window is dropped"),
    ("scripts/prepare_skeletons.py", "def le2i_fall_frames",
     "Le2i has no fall/ADL marker in any path component; without this its ~192 fall "
     "clips land in the negatives (notebook 02)"),
    ("scripts/prepare_skeletons.py", "def label_fall_windows",
     "shared fall labelling: exact interval for Le2i, positional fallback elsewhere"),
    ("src/behaviorsense/data/skeleton_dataset.py", "keep_root_motion",
     "falls are unlearnable without it"),
    ("src/behaviorsense/models/ensemble.py", "def per_stream_logits", "notebook 04 ablation"),
    ("src/behaviorsense/models/stgcnpp.py", "parent.setdefault",
     "flip-equivariant bone stream; without it half the ensemble trains on sign noise"),
    ("src/behaviorsense/kaggle_artifacts.py", "def find_run_dir",
     "one rule for 'is this real session output or a dev leftover'; four call sites "
     "learned it separately and the fourth was missed"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def repair_claim",
     "format-only claim repair + maxItems bound to max_claims (notebook 04). Without "
     "it the unconstrained arm scores 245 emitted / 0 scorable / nan%, which measures "
     "JSON compliance rather than faithfulness"),
    ("scripts/eval_hallucination.py", "unusable_rate",
     "three-arm hallucination table with a denominator over EMITTED claims; a stale "
     "snapshot silently reports the two-arm nan% version"),
    ("src/behaviorsense/pipeline.py", "def frames_to_windows", "Agent 1 -> Agent 2 seam"),
    ("configs/taxonomy.yaml",        None,             "class map (notebook 01)"),
]
_stale = []
for _rel, _token, _why in CONTRACT:
    _p = CODE / _rel
    if not _p.is_file():
        _stale.append(f"{_rel} is MISSING ({_why})")
        continue
    if _token and _token not in _p.read_text(encoding="utf-8", errors="ignore"):
        _stale.append(f"{_rel} lacks {_token!r} ({_why})")
if _stale:
    raise AssertionError(
        "The attached code dataset is OLDER than these notebooks:\n  - "
        + "\n  - ".join(_stale)
        + f"\n\nMounted: {CODE}\nRe-upload from your checkout, then restart this notebook:"
          "\n  kaggle datasets version -p . --dir-mode zip -m \"sync\""
    )
print(f"  contract {len(CONTRACT)}/{len(CONTRACT)} - mounted code is current")

In [ ]:
# Prove the GPU is actually in use BEFORE spending the session.
#
# onnxruntime falls back to CPU silently: device="cuda" against a CPU-only build, or a CUDA
# build whose provider .so fails to load, both yield correct poses at a fraction of the
# speed. That is a 12-hour session extracting a fraction of one slice, and it presents as
# "the GPU is slow" rather than "the GPU is not being used".
import onnxruntime as ort
import importlib.metadata as _md

providers = ort.get_available_providers()
print("onnxruntime providers:", providers)
_installed = {d.metadata["Name"].lower() for d in _md.distributions()
              if d.metadata.get("Name")}
_both = {"onnxruntime", "onnxruntime-gpu"} & _installed
_hint = ("BOTH onnxruntime builds are installed: they share the `onnxruntime` module "
         "directory, so whichever pip wrote last wins. Re-run the setup cell - it "
         "uninstalls the CPU build and installs rtmlib with --no-deps."
         if len(_both) == 2 else
         "Check that notebook 00 staged onnxruntime-gpu (not onnxruntime).")
assert "CUDAExecutionProvider" in providers, (
    f"onnxruntime has no CUDA provider - extraction would run on CPU. "
    f"available={providers}, packages={sorted(_both) or 'none'}. {_hint}")
print(f"CUDA provider compiled in (packages: {sorted(_both)})")

In [ ]:
# Build the pose model from the STAGED onnx, and prove it works before the loop.
#
# This notebook has no internet, so rtmlib must not be left to fetch anything: the
# constructor is given the explicit path to the rtmo-l.onnx that notebook 00 staged.
# Passing mode= instead would have rtmlib resolve a URL, which offline either hangs or
# raises - and notebook 00 asserts this file exists precisely because 01/02 need it, so
# leaving it unused was a staged asset nothing consumed.
#
# The probe below runs ONE synthetic frame. A five-second failure here is the whole point:
# a wrong kwarg or a bad onnx otherwise surfaces at the first real video, after the unzip.
import numpy as np
from rtmlib import RTMO

_onnx = sorted(pathlib.Path("/kaggle/input").glob("**/rtmo-l.onnx"))
assert _onnx, (
    "rtmo-l.onnx not found under /kaggle/input. Attach the dataset notebook 00 produced "
    f"(wheels+weights, e.g. behavioursense-WW). Mounted: {ATTACHED}")
RTMO_ONNX = _onnx[0]
print(f"  rtmo    {RTMO_ONNX}  ({RTMO_ONNX.stat().st_size/1e6:.0f} MB)")

body = RTMO(onnx_model=str(RTMO_ONNX), model_input_size=(640, 640),
            backend="onnxruntime", device="cuda")
_k, _s = body(np.zeros((480, 640, 3), np.uint8))
print(f"  probe OK - keypoints {np.asarray(_k).shape}, scores {np.asarray(_s).shape}")

# get_available_providers() lists what the build was COMPILED with, not what LOADED.
# Observed: onnxruntime-gpu 1.28 (a CUDA 13 build) on Kaggle's CUDA 12 image lists
# CUDAExecutionProvider, then logs "Failed to load library ... libcublasLt.so.13" and
# runs the model on CPU anyway. The cell above passes; the session is ~50x too slow.
# The session object is the only honest source: it reports the providers in EFFECT.
_sess = getattr(body, "session", None)
_active = list(_sess.get_providers()) if _sess is not None else []
assert "CUDAExecutionProvider" in _active, (
    f"RTMO is running on {_active or 'an unknown provider'} - the CUDA provider was "
    "listed but did NOT load. Scroll up for a 'Failed to load library' line naming the "
    "missing .so: libcublasLt.so.13 means an onnxruntime-gpu built for CUDA 13 on this "
    "CUDA 12 image. Notebook 00 pins onnxruntime-gpu==1.26.0 (the newest CUDA 12.8 "
    "build) - if this cache predates that pin, re-run notebook 00.")
print(f"  provider IN USE: {_active[0]}")

In [ ]:
# Locate this session's slice of Charades. TWO layouts are possible and which one you
# get is Kaggle's choice, not yours:
#
#   PRE-EXTRACTED  Kaggle auto-unpacks archives when building a dataset from notebook
#                  output, so 01a's Charades_v1_480.zip arrives as 9,848 loose mp4s under
#                  <mount>/Charades_v1_480/Charades_v1_480/. This is the better case:
#                  read the files in place, no unpack, nothing in the 20 GB working quota.
#   ZIPS           the archives survived as .zip (direct upload, or auto-extract off).
#                  Unpacking all of them costs ~13 GB and minutes of every session, so the
#                  annotation CSV fixes the video order and only this slice is extracted.
#
# Assuming zips cost a session: the assert read "charades-480p is not attached" while it
# was plainly attached and extracted. So detect, and say which layout was found.
import zipfile, pathlib, csv, time

INPUT = pathlib.Path("/kaggle/input")
ATTACHED_NOW = sorted(p.name for p in INPUT.iterdir()) if INPUT.is_dir() else []
_ann_csv = sorted(INPUT.glob("**/Charades_v1_train.csv"))
_mp4     = sorted(INPUT.glob("**/Charades_v1_480/**/*.mp4"))[:1]
_zips    = sorted(INPUT.glob("**/Charades*.zip"))
assert _ann_csv or _zips, (
    "Charades is not attached in either layout: no **/Charades_v1_train.csv (extracted) "
    f"and no **/Charades*.zip (archived) under /kaggle/input. Attached: {ATTACHED_NOW}. "
    "Run notebook 01a and attach its output dataset.")

TMP = pathlib.Path("/tmp/charades"); TMP.mkdir(parents=True, exist_ok=True)
if _ann_csv:
    ann_csv = _ann_csv[0]
    print(f"layout PRE-EXTRACTED (Kaggle unpacked the archives)")
else:
    with zipfile.ZipFile(next(p for p in _zips if "annotation" in p.name.lower())) as z:
        z.extractall(TMP / "annotations")
    ann_csv = next((TMP / "annotations").rglob("Charades_v1_train.csv"))
    print(f"layout ZIPS ({len(_zips)} archives)")

rows = list(csv.DictReader(open(ann_csv, encoding="utf-8")))[VID_START:VID_END]
wanted = {r["id"] for r in rows}
assert wanted, (f"slice {VID_START}..{VID_END} selected 0 videos from {ann_csv.name} "
                f"({len(list(csv.DictReader(open(ann_csv, encoding='utf-8'))))} rows total)")
print(f"  slice {VID_START}..{VID_START + len(rows)} -> {len(wanted)} video ids")

t0 = time.time()
if _mp4:
    # Read in place. VIDEO_DIRS is what the extraction cell resolves paths against.
    VIDEO_DIRS = sorted({p.parent for p in INPUT.glob("**/Charades_v1_480/**/*.mp4")})
    n_vid = sum(len(list(d.glob("*.mp4"))) for d in VIDEO_DIRS)
    print(f"  {n_vid} mp4s readable in place across {len(VIDEO_DIRS)} dir(s) - no unpack")
else:
    VIDEOS = TMP / "videos"; VIDEOS.mkdir(exist_ok=True)
    with zipfile.ZipFile(next(p for p in _zips if "480" in p.name)) as z:
        members = [n for n in z.namelist()
                   if n.endswith(".mp4") and pathlib.Path(n).stem in wanted]
        for i, m in enumerate(members):
            target = VIDEOS / pathlib.Path(m).name
            if not target.exists():
                with z.open(m) as src, open(target, "wb") as dst:
                    dst.write(src.read())
            if (i + 1) % 500 == 0:
                print(f"  {i+1}/{len(members)} extracted")
    VIDEO_DIRS = [VIDEOS]
    n_vid = len(list(VIDEOS.glob("*.mp4")))
    print(f"  {n_vid} videos unpacked in {(time.time()-t0)/60:.1f} min")
assert n_vid, f"0 videos found for this slice - layout unexpected. Attached: {ATTACHED_NOW}"

# One resolver for both layouts, so the extraction cell needs no branch of its own.
def video_path(vid):
    for d in VIDEO_DIRS:
        p = d / f"{vid}.mp4"
        if p.is_file():
            return p
    return None
_hit = sum(video_path(v) is not None for v in wanted)
assert _hit, (f"none of the {len(wanted)} slice ids resolved to a file under "
              f"{[str(d) for d in VIDEO_DIRS]} - id/filename mismatch")
print(f"  {_hit}/{len(wanted)} slice ids resolve to a readable mp4")

In [ ]:
# Build + REVIEW the class map. The fallback list printed here is the eyeball check the
# taxonomy demands — label noise from silent auto-mapping is unrecoverable after training.
import subprocess, glob
# Both layouts: the extracted dataset has it under the mount, the zip layout under /tmp.
_cls = (sorted(pathlib.Path("/kaggle/input").glob("**/Charades_v1_classes.txt"))
        or sorted(pathlib.Path("/tmp/charades").rglob("Charades_v1_classes.txt")))
assert _cls, "Charades_v1_classes.txt not found in either layout"
classes_txt = str(_cls[0])
subprocess.run(
    ["python", str(SCRIPTS / "build_charades_map.py"),
     "--classes", classes_txt,
     "--out", "/kaggle/working/charades_map.yaml",
     "--review-tsv", "/kaggle/working/charades_map_review.tsv"],
    check=True)

In [ ]:
# Carry forward shards from the previous dataset version (continuation runs).
#
# This cell is where a 12-hour session gets destroyed if it is sloppy. Kaggle dataset
# versions REPLACE content — they do not merge. So if a previous version exists and this
# cell fails to copy it forward, Save Version publishes only the current slice and the
# earlier extraction is gone. "No shards found" therefore has two very different meanings:
#   - dataset not attached at all      -> genuinely the first run, proceed
#   - dataset attached but nothing in it -> wrong path or wrong version. REFUSE. Printing
#     "fresh start" here and carrying on is how prior work gets overwritten.
import shutil, pathlib
OUT = pathlib.Path("/kaggle/working/shards"); OUT.mkdir(exist_ok=True)

# Find the shards mount by name, tolerating the behaviour/behavior spelling this project
# has already been uploaded under. A hard-coded name that misses is indistinguishable from
# "not attached", which is precisely the branch that destroys prior work.
MOUNTS = [m for m in sorted(pathlib.Path("/kaggle/input").iterdir())
          if "adl" in m.name.lower().replace("behaviour", "behavior")]          if pathlib.Path("/kaggle/input").is_dir() else []

if not MOUNTS:
    print("fresh start - no ADL shards dataset attached")
else:
    # Attached. Find the shards wherever they sit, rather than assuming one layout.
    found = sorted(p for m in MOUNTS for p in m.rglob("*.npz"))
    assert found, (
        f"an ADL shards dataset IS attached ({[m.name for m in MOUNTS]}) but contains no "
        f".npz. Saving now would REPLACE the dataset with only this session's slice and "
        f"destroy the previous extraction.")
    for f in found:
        shutil.copy(f, OUT / f.name)
    print(f"carried forward {len(found)} shard(s) from {found[0].parent}")

In [ ]:
# Pose extraction. Top-2 people by box area per frame -> [T, 2, 17, 3] float16.
# Window labels: the Charades action interval covering >= 60% of the window, mapped
# through the reviewed YAML; unmapped -> other_idle; drop:true classes excluded.
# Subject ids: Charades publishes no worker ids, so the VIDEO id is the subject proxy
# and the P1 split is by video — stated in the eval tables, not hidden.
import csv, glob, time, yaml, pathlib
import numpy as np
import cv2
from prepare_skeletons import assign_slots, window_clip   # tested repo code, not copies

mapping = yaml.safe_load(open("/kaggle/working/charades_map.yaml"))["charades"]
tax = yaml.safe_load(open(CONFIGS / "taxonomy.yaml"))
name_to_id = {c["name"]: c["id"] for c in tax["classes"]}

# rows and video_path() were resolved by the layout cell above - do not re-derive them
# here, or this cell silently disagrees with the one that verified the slice exists.
print(f"videos {VID_START}..{VID_START + len(rows)} ({len(VIDEO_DIRS)} source dir(s))")

skels, labels, subjects, sources = [], [], [], []
shard_idx = len(list(OUT.glob("*.npz")))
t0 = time.time()

def flush():
    global skels, labels, subjects, sources, shard_idx
    if not skels:
        return
    np.savez_compressed(
        OUT / f"charades_{shard_idx:04d}.npz",
        skeletons=np.stack(skels).astype(np.float16),
        labels=np.asarray(labels, dtype=np.int64),
        subjects=np.asarray(subjects, dtype="<U32"),
        datasets=np.asarray(sources, dtype="<U32"))
    print(f"  shard {shard_idx}: {len(skels)} windows")
    shard_idx += 1
    skels, labels, subjects, sources = [], [], [], []

for n, row in enumerate(rows):
    vid = row["id"]
    _p = video_path(vid)
    if _p is None:
        continue
    path = str(_p)
    cap = cv2.VideoCapture(path)
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 24
    step = max(1, round(src_fps / FPS_SAMPLE))
    poses, f = [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if f % step == 0:
            kpts, scores = body(frame)
            # Slot assignment tracks the previous frame (assign_slots): area order alone
            # swaps the two people whenever their apparent sizes cross, splicing both
            # trajectories and teleporting the motion streams at the swap frame.
            poses.append(assign_slots(kpts, scores, poses[-1] if poses else None))
        f += 1
    cap.release()
    if len(poses) < 30:
        continue
    poses = np.stack(poses)

    acts = []
    for a in (row["actions"] or "").split(";"):
        if a.strip():
            cid, s, e = a.split()
            acts.append((cid, float(s), float(e)))
    for w_i, w in enumerate(window_clip(poses)):
        ws, we = w_i * 1.0, w_i * 1.0 + 2.0
        # Best-covering mapped action wins the window. The first version broke on the
        # FIRST action with >= 60% overlap in CSV order - Charades actions overlap
        # heavily, so a barely-qualifying interval listed first beat a fully-covering
        # one, and an UNMAPPED first qualifier forced other_idle even when a mapped
        # action also covered the window. Coverage decides now; unmapped actions cannot
        # claim a window at all.
        lab, best_ov = "other_idle", 0.0
        for cid, s, e in acts:
            ov = min(we, e) - max(ws, s)
            if ov < 1.2 or ov <= best_ov:                 # >= 60% of the 2 s window
                continue
            slugs = [k for k in mapping if k.startswith(cid + "_")]
            m = mapping.get(slugs[0]) if slugs else None
            if m is None:
                continue
            best_ov, lab = ov, (None if isinstance(m, dict) else m)  # dict == drop:true
        if lab is None:
            continue
        skels.append(w)
        labels.append(name_to_id[lab])
        subjects.append(vid)
        sources.append("charades")
    if sum(s.nbytes for s in skels) > 400e6:
        flush()
    if n % 100 == 0 and n:
        rate = n / max(time.time() - t0, 1)
        eta_h = (len(rows) - n) / max(rate, 1e-9) / 3600
        print(f"{n}/{len(rows)} videos ({rate * 3600:.0f}/h, ETA {eta_h:.1f} h)")
flush()

In [ ]:
# Quality gates — printed, not assumed. A class distribution that surprises you here is
# cheaper than one that surprises you after 80 epochs.
import numpy as np, pathlib, collections, csv
OUT = pathlib.Path("/kaggle/working/shards")
counts, n_windows = collections.Counter(), 0
for f in OUT.glob("*.npz"):
    z = np.load(f)
    n_windows += len(z["labels"])
    counts.update(z["labels"].tolist())
print(f"{n_windows} windows across {len(list(OUT.glob('*.npz')))} shards")
for lab, n in counts.most_common():
    print(f"  class {lab:>2}: {n:>7}  ({n / max(n_windows, 1):.1%})")
print()
# Never invite a Save Version that would publish an empty dataset over a good one.
assert n_windows > 0, ("0 windows extracted - do NOT Save Version, it would replace "
                       "behaviorsense-adl-shards with nothing. Check VID_START/VID_END "
                       "and that the Charades videos actually downloaded.")
print("Save Version (QUICK SAVE - not 'Save & Run All', which re-extracts from scratch)")
print("  -> update dataset behaviorsense-adl-shards from /kaggle/working.")
# Report completion against the SPLIT, not against VID_END. With VID_END=99999 (meaning
# "to the end") the old hint said "next session: VID_START=99999", which is both wrong
# and alarming. What matters is whether every row of the split was actually consumed.
_done = VID_START + len(rows)
_total = len(list(csv.DictReader(open(ann_csv, encoding="utf-8"))))
if _done >= _total:
    print(f"COMPLETE: {_done}/{_total} videos of the train split extracted. "
          "No continuation session needed - go to notebook 02.")
else:
    print(f"PARTIAL: {_done}/{_total} videos. Next session set VID_START={_done} "
          "and attach this output dataset as an input so its shards carry forward.")